## 🎬 Bab 2: Dari Data Bersih ke Mesin Pencari "Kembaran"

Di Bab 1, kita sudah membersihkan data sampai tersisa **1.464 pemain lapangan** yang layak dibandingkan. Sekarang saatnya menjawab pertanyaan utama sang direktur olahraga:

> **"Kalau pemain ini pergi, siapa yang gaya mainnya paling mirip?"**

Idenya sederhana. Setiap pemain kita ubah menjadi **sidik jari gaya main**, yaitu deretan angka yang menggambarkan caranya menembak, mengumpan, bertahan, dan beradu fisik. Pemain yang sidik jarinya paling dekat adalah calon penggantinya.

| Babak | Pertanyaan yang dijawab | Alat |
|---|---|---|
| 📥 Memuat data | Apakah hasil Bab 1 sampai utuh? | `read_parquet` |
| 🧬 Memilih fitur | Angka apa yang membentuk sidik jari? | 12 fitur |
| ⚖️ Menyamakan skala | Bagaimana agar semua fitur didengar setara? | `StandardScaler` |
| 🔍 Mencari tetangga | Bagaimana mengukur "mirip"? | `NearestNeighbors` + cosine |
| 💾 Menyimpan | Apa yang dibawa ke dashboard? | `joblib` |
| 📏 Menguji | Seberapa mirip hasil rekomendasinya? | Cosine distance top-5 |

### 🧰 Menyiapkan Peralatan

Empat alat yang kita bawa di bab ini:

- **`pandas`** untuk membaca dan mengolah tabel pemain.
- **`StandardScaler`** untuk menyamakan skala antar fitur.
- **`NearestNeighbors`** untuk mencari pemain dengan profil terdekat.
- **`joblib`** untuk menyimpan model agar bisa dipakai kembali oleh dashboard tanpa melatih ulang.

In [8]:
# Cell 1: Import Library
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import joblib
from pathlib import Path
import os

In [9]:
# Cell 2: Load Data yang Sudah Dibersihkan
# Kita panggil file parquet yang sudah dibuat dari notebook 01
data_path = "../data/processed/players_cleaned.parquet"
df = pd.read_parquet(data_path)

print(f"Data terbaca dengan ukuran: {df.shape}")
df.head(3)

Data terbaca dengan ukuran: (1464, 60)


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,PKA,PKsv,PKm,Gls_per90,Ast_per90,Crs_per90,TklW_per90,Int_per90,Fld_per90,Fls_per90
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,NaN,NaN,NaN,0.147059,0.183824,1.360294,0.992647,0.625000,1.875000,0.735294
1,6,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,25.0,1999.0,13,11,...,NaN,NaN,NaN,0.190476,0.000000,0.857143,1.619048,1.238095,1.142857,1.523810
2,7,Ali Abdi,tn TUN,"MF,DF",Nice,fr Ligue 1,31.0,1993.0,22,13,...,NaN,NaN,NaN,0.227273,0.000000,2.500000,0.833333,0.757576,1.363636,1.666667


**Hasil:** data terbaca dengan ukuran **(1.464, 60)**. Angka 60 berasal dari 53 kolom asli ditambah 7 kolom `_per90` yang kita buat di Bab 1, jadi serah terima antar notebook berjalan mulus.

Perhatikan juga kolom `Rk` yang melompat (1, 6, 7). Baris-baris di antaranya adalah kiper dan pemain bermenit minim yang sudah tersaring. Jerome Abbey yang kita temui di awal Bab 1 sudah tidak ada di sini.

---
### 🧬 Memilih Sidik Jari: 12 Fitur Gaya Main

Dari 60 kolom, tidak semuanya menggambarkan *gaya main*. Kolom seperti `Rk`, `Born`, atau `Squad` tidak bercerita tentang cara seseorang bermain. Kita hanya memilih **12 fitur** yang mewakili beberapa sisi permainan:

| Sisi permainan | Fitur |
|---|---|
| 🧓 Profil pemain | `Age` |
| 🎯 Menembak | `Sh/90`, `SoT/90`, `SoT%`, `G/Sh` |
| ⚽ Output serangan | `Gls_per90`, `Ast_per90`, `Crs_per90` |
| 🛡️ Bertahan | `TklW_per90`, `Int_per90` |
| 💥 Duel fisik | `Fld_per90`, `Fls_per90` |

Satu catatan teknis: pemain yang tidak pernah menembak punya nilai kosong di `SoT%` dan `G/Sh`. Kolom kosong ini diisi **0** supaya model tidak berhenti di tengah jalan.

In [10]:
# Cell 3: Pilih Fitur (Features Selection)
# Ini adalah metrik yang akan dipakai model untuk mencari kemiripan
features = [
    'Age', 'Sh/90', 'SoT/90', 'SoT%', 'G/Sh',
    'Gls_per90', 'Ast_per90', 'Crs_per90', 
    'TklW_per90', 'Int_per90', 'Fld_per90', 'Fls_per90'
]

# Ambil hanya kolom fitur tersebut dari dataset
X = df[features].copy()

# Cek apakah ada nilai kosong (NaN), jika ada isi dengan 0
X = X.fillna(0)
print("Fitur siap distandardisasi.")

Fitur siap distandardisasi.


**Fitur siap.** Namun ada jebakan yang harus dihindari sebelum kita mengukur kemiripan.

---
### ⚖️ Menyamakan Skala: Meter vs Gram

Lihat perbedaan skalanya. `Age` berkisar di angka 20-an sampai 30-an, sedangkan `Gls_per90` hanya berkisar antara 0 sampai sekitar 1. Kalau dibiarkan, jarak antar pemain akan **didominasi oleh umur**, dan gol per 90 menit nyaris tidak terdengar. Ini seperti membandingkan dua benda dengan menjumlahkan meter dan gram.

`StandardScaler` menyelesaikannya dengan mengubah setiap fitur menjadi **rata-rata 0 dan variansi 1**. Dengan begitu 12 fitur punya "volume suara" yang setara.

In [11]:
# Cell 4: Standarisasi Data (StandardScaler)
# Membuat skala data seragam (mean = 0, variansi = 1) agar semua metrik berimbang
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data berhasil distandardisasi.")

Data berhasil distandardisasi.


Semua fitur kini berbicara dengan volume yang sama. Saatnya membangun mesin pencarinya.

---
### 🔍 Mencari Tetangga Terdekat dengan Cosine

**`NearestNeighbors` tidak "belajar" apa pun.** Tidak ada label benar atau salah di sini (itulah arti *unsupervised*). Perintah `fit` hanya menyimpan 1.464 sidik jari, lalu ketika kita bertanya "siapa yang paling mirip dengan pemain X?", model mencari vektor yang paling dekat.

Ukuran dekatnya memakai **cosine distance**:

$$\text{cosine distance} = 1 - \cos(\theta)$$

Artinya, dua pemain dianggap mirip kalau *arah* profil mereka searah. Nilai **0** berarti identik dan makin besar berarti makin berbeda.

> 📝 **Catatan:** karena data sudah distandardisasi, cosine membandingkan *bentuk* profil, bukan besar angkanya. Dua pemain dengan pola yang sama tetapi intensitas berbeda bisa tampak sangat mirip.

In [12]:
# Cell 5: Latih Model NearestNeighbors
# Kita pakai metrik 'cosine' yang sangat bagus untuk sistem rekomendasi / mencari kemiripan pola
# n_neighbors kita set cukup besar (misal 10) buat opsi
nn_model = NearestNeighbors(n_neighbors=10, metric='cosine')
nn_model.fit(X_scaled)

print("Model NearestNeighbors berhasil dilatih!")

Model NearestNeighbors berhasil dilatih!


Model sudah terbentuk. Sekarang jangan sampai hilang saat notebook ditutup.

---
### 💾 Menyimpan Model dan Scaler: Satu Paket

Kita menyimpan **dua** file, bukan satu, karena keduanya satu paket:

- **`scaler.joblib`** adalah penerjemah. Saat pengguna dashboard memilih seorang pemain, angkanya harus diubah dengan skala yang *persis sama* seperti saat model dilatih.
- **`nn_model.joblib`** adalah mesin pencarinya.

Dashboard Streamlit di Bab berikutnya cukup memuat kedua file ini, tanpa melatih ulang apa pun.

In [13]:
# Cell 6: Simpan Model & Scaler dengan joblib
# Buat folder models jika belum ada
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Simpan objek scaler dan model ke file
joblib.dump(scaler, models_dir / "scaler.joblib")
joblib.dump(nn_model, models_dir / "nn_model.joblib")

print("scaler.joblib dan nn_model.joblib berhasil disimpan di folder models!")# Cell 6: Simpan Model & Scaler dengan joblib
# Buat folder models jika belum ada
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Simpan objek scaler dan model ke file
joblib.dump(scaler, models_dir / "scaler.joblib")
joblib.dump(nn_model, models_dir / "nn_model.joblib")

print("scaler.joblib dan nn_model.joblib berhasil disimpan di folder models!")

scaler.joblib dan nn_model.joblib berhasil disimpan di folder models!
scaler.joblib dan nn_model.joblib berhasil disimpan di folder models!


---
### 📏 Menguji: Seberapa Mirip Rekomendasinya?

Karena tidak ada kunci jawaban, kita mengevaluasinya lewat **jarak itu sendiri**. Untuk setiap pemain, kita ambil 6 tetangga terdekat. Tetangga pertama (indeks 0) adalah **dirinya sendiri**, jadi kita buang. Sisanya adalah **top-5 rekomendasi pengganti**, lalu kita rangkum jaraknya untuk seluruh 1.464 pemain.

In [14]:
# Ambil tetangga terdekat untuk SELURUH pemain di dataset sekaligus
# n_neighbors=6 -> index 0 adalah diri sendiri, 1 s/d 5 adalah top-5 rekomendasi
all_distances, all_indices = nn_model.kneighbors(X_scaled, n_neighbors=6)

# Ambil distance dari top-5 tetangga (abaikan kolom pertama/indeks 0)
top5_distances = all_distances[:, 1:]

# 1. Rata-rata Cosine Distance global
global_mean_distance = np.mean(top5_distances)
global_min_distance = np.min(top5_distances)
global_max_distance = np.max(top5_distances)

# 2. Rata-rata Similarity Score global
global_mean_similarity = (1 - global_mean_distance) * 100

print("=" * 50)
print("📊 GLOBAL EVALUATION METRICS (Seluruh Pemain)")
print("=" * 50)
print(
    f"Rata-rata Kemiripan Top-5      : {global_mean_similarity:.2f}%"
)
print(
    f"Rata-rata Cosine Distance      : {global_mean_distance:.4f} (Makin kecil makin identik)"
)
print(f"Jarak Terdekat (Best Match)    : {global_min_distance:.4f}")
print(f"Jarak Terjauh di Top-5         : {global_max_distance:.4f}")
print("=" * 50)

NameError: name 'np' is not defined

**Cara membaca hasilnya:**

| Metrik | Nilai | Artinya |
|---|---|---|
| Rata-rata cosine distance | **0,1368** | Rata-rata rekomendasi cukup dekat dengan pemain aslinya |
| Rata-rata kemiripan | **86,32%** | Hanya `1 - rata-rata jarak`, bukan akurasi model |
| Jarak terdekat | **0,0042** | Ada pasangan pemain yang profilnya nyaris kembar |
| Jarak terjauh (top-5) | **0,4455** | Ada pemain unik yang rekomendasinya kurang kuat |

⚠️ **Perlu dipahami:** angka 86,32% adalah ukuran *kedekatan*, bukan *ketepatan*. Tanpa label kebenaran, kita belum bisa menyatakan bahwa rekomendasinya "benar", hanya bahwa rekomendasinya **dekat secara statistik**. Penilaian terakhir tetap perlu mata seorang pemandu bakat.

---
## 🏁 Epilog: Mesin Pencari Sudah Siap

Ringkasan Bab 2:

- Kita memuat **1.464 pemain** hasil Bab 1 dan memilih **12 fitur** sebagai sidik jari gaya main.
- Kita menyamakan skala dengan **`StandardScaler`** agar tidak ada fitur yang mendominasi.
- Kita membangun mesin pencari dengan **`NearestNeighbors` (cosine)** dan menyimpannya bersama scaler.
- Rata-rata kemiripan top-5 sebesar **86,32%** menunjukkan rekomendasinya cukup rapat, dengan beberapa pemain unik sebagai pengecualian.

**Selanjutnya (Bab 3):** kedua file di folder `models/` akan dimuat oleh **dashboard Streamlit**. Pengguna cukup memilih seorang pemain, dan sistem menampilkan kandidat penggantinya.